# 01 — The frozen disc localiser, and the crops every label source will share

**Runtime → Change runtime type → GPU → L4.** An A100 buys nothing here: this is one short
training run and then a lot of disk writing.

Audit fix A1. If the crop moves when the convention changes, a model trained on a liberal
expert is also looking at a different field of view, and nothing downstream can separate the
two. So one localiser is trained **once**, on the official merge only, then frozen; it emits
one box per eye; and all nine label sources are cropped with that same box.

Containment of all seven experts' discs is then **proven on every eye**, not assumed.

About 30–45 minutes. It resumes if Colab drops you.


In [ ]:
# --- environment -----------------------------------------------------------
# WHERE THINGS LIVE, and why:
#   PROJECT  this project's code (src/, analysis/)   -> Drive, or a GitHub clone
#   RAW      the 1.5 GB dataset                      -> LOCAL disk: Drive I/O is slow,
#                                                       and re-fetching costs ~3 minutes
#   WORK     crops, weights, results                 -> Drive, so a disconnect costs nothing
REPO_URL = "https://github.com/Aduitya9/Opiotica.git"   # cloned fresh each session
DRIVE_PROJECT = "/content/drive/MyDrive/whose-eyes"     # offline fallback if REPO_URL is ""
WORK = "/content/drive/MyDrive/whose-eyes-work"
RAW = "/content/raw"

import json, os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")

if REPO_URL:
    if not os.path.isdir("/content/project"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/project"], check=True)
    PROJECT = "/content/project"
else:
    PROJECT = DRIVE_PROJECT
assert os.path.isdir(os.path.join(PROJECT, "src")), (
    f"No src/ under {PROJECT}. Copy the `whose-eyes` folder into your Drive at "
    f"{DRIVE_PROJECT}, or set REPO_URL above.")
sys.path.insert(0, PROJECT)
for sub in ("crops", "runs", "results"):
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)
os.makedirs(RAW, exist_ok=True)

# --- GPU ---------------------------------------------------------------------
# Batch size is set from the card, not guessed: U-Net + EfficientNet-B4 at 512 px with AMP
# needs roughly 1.5 GB per sample. Everything else in the schedule stays fixed.
import torch

if torch.cuda.is_available():
    GPU = torch.cuda.get_device_name(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH = 24 if VRAM > 60 else 16 if VRAM > 35 else 8 if VRAM > 20 else 4
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    GPU, VRAM, BATCH = "cpu", 0.0, 2

print(f"project : {PROJECT}")
print(f"work    : {WORK}   (Drive - survives a disconnect)")
print(f"raw     : {RAW}   (local - re-fetched each session)")
print(f"gpu     : {GPU}  {VRAM:.0f} GB  ->  batch size {BATCH}")


In [ ]:
!pip -q install "segmentation-models-pytorch==0.4.0" "timm>=1.0.9" || pip -q install segmentation-models-pytorch

import glob, hashlib, time
import numpy as np, pandas as pd
from PIL import Image
import segmentation_models_pytorch as smp

from src import refuge, crops as crops_mod, labels as labels_mod, train as train_mod

RELEASE = sorted({os.path.dirname(p) for p in
                  glob.glob(os.path.join(RAW, "**", "Training-400"), recursive=True)})[0]
inventory = pd.read_csv(os.path.join(WORK, "results", "inventory.csv"), dtype={"image_id": str})
png = pd.read_csv(os.path.join(WORK, "results", "png_measurements.csv"), dtype={"image_id": str})
MARGIN = json.load(open(os.path.join(WORK, "results", "crop_margin.json")))["margin"]
folder_of = {i: f for _, i, f in refuge.iter_images(RELEASE)}
print(f"{len(inventory)} eyes | frozen margin {MARGIN} | batch {BATCH} on {GPU}")


In [ ]:
# --- the fixed split, drawn once and saved -----------------------------------
# 320 Zeiss eyes to train on, 80 held back for early stopping, 800 Canon eyes to evaluate on.
# Byte-identical for every label source, for the whole study.
SPLIT_FILE = os.path.join(WORK, "results", "zeiss_train80_val20.json")
if os.path.exists(SPLIT_FILE):
    split_ids = json.load(open(SPLIT_FILE))
    print("reusing the existing split - it is never redrawn")
else:
    zeiss = inventory[inventory.camera == "Zeiss"].image_id.tolist()
    shuffled = list(np.random.default_rng(20261113).permutation(zeiss))
    split_ids = {"train": sorted(shuffled[:320]), "val": sorted(shuffled[320:]),
                 "eval": sorted(inventory[inventory.camera == "Canon"].image_id)}
    json.dump(split_ids, open(SPLIT_FILE, "w"), indent=2)
    print("split drawn and frozen")
print({k: len(v) for k, v in split_ids.items()})


In [ ]:
# --- localiser dataset: whole image -> official disc only --------------------
LOC_SIZE = 512

class LocaliserSet(torch.utils.data.Dataset):
    def __init__(self, ids, augment=False):
        self.ids, self.augment = list(ids), augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, k):
        image_id = self.ids[k]
        folder = folder_of[image_id]
        jpg = next(f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg")))
        image = Image.open(os.path.join(folder, jpg)).convert("RGB").resize((LOC_SIZE, LOC_SIZE))
        _, disc = refuge.official_masks(folder, image_id)
        disc = np.array(Image.fromarray(disc.astype(np.uint8) * 255)
                        .resize((LOC_SIZE, LOC_SIZE), Image.NEAREST)) > 127
        x = np.asarray(image, dtype=np.float32).transpose(2, 0, 1) / 255.0
        y = disc[None].astype(np.float32)
        if self.augment and np.random.rand() < 0.5:
            x, y = x[:, :, ::-1].copy(), y[:, :, ::-1].copy()
        return torch.from_numpy(x), torch.from_numpy(y)

print(len(LocaliserSet(split_ids["train"])), "training eyes for the localiser")


In [ ]:
# --- train the localiser ONCE, then freeze it --------------------------------
LOC_DIR = os.path.join(WORK, "runs", "localiser")
os.makedirs(LOC_DIR, exist_ok=True)
WEIGHTS = os.path.join(LOC_DIR, "best.pt")

localiser = smp.Unet("efficientnet-b4",
                     encoder_weights=None if os.path.exists(WEIGHTS) else "imagenet",
                     in_channels=3, classes=1)

if os.path.exists(WEIGHTS):
    localiser.load_state_dict(torch.load(WEIGHTS, weights_only=True))
    localiser = localiser.cuda().eval()
    print("localiser already trained - loading the frozen weights, not retraining")
else:
    train_mod.set_seed(20261113)
    localiser = localiser.cuda()
    opt = torch.optim.AdamW(localiser.parameters(), lr=3e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    loaders = {
        "train": torch.utils.data.DataLoader(LocaliserSet(split_ids["train"], augment=True),
                                             batch_size=BATCH, shuffle=True, num_workers=2,
                                             drop_last=True, pin_memory=True),
        "val": torch.utils.data.DataLoader(LocaliserSet(split_ids["val"]), batch_size=BATCH,
                                           num_workers=2, pin_memory=True),
    }
    best = float("inf")
    for epoch in range(30):
        line = {}
        for phase in ("train", "val"):
            localiser.train(phase == "train")
            total = count = 0
            with torch.set_grad_enabled(phase == "train"):
                for x, y in loaders[phase]:
                    x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
                    with torch.amp.autocast("cuda"):
                        loss = train_mod.dice_bce(localiser(x), y)
                    if phase == "train":
                        opt.zero_grad(set_to_none=True)
                        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                    total += loss.item() * x.size(0); count += x.size(0)
            line[phase] = total / count
        marker = ""
        if line["val"] < best:
            best, marker = line["val"], "  *"
            torch.save(localiser.state_dict(), WEIGHTS)
        print(f"epoch {epoch:2d}  train {line['train']:.4f}  val {line['val']:.4f}{marker}")
    localiser.load_state_dict(torch.load(WEIGHTS, weights_only=True))
    localiser.eval()

for p in localiser.parameters():
    p.requires_grad = False
print("\nFROZEN. This model is never trained again in this project.")


In [ ]:
# --- one box per eye, from the frozen localiser ------------------------------
all_ids = inventory.image_id.tolist()
rows, failures = [], []

with torch.no_grad():
    for n, image_id in enumerate(all_ids, 1):
        folder = folder_of[image_id]
        jpg = next(f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg")))
        full = Image.open(os.path.join(folder, jpg)).convert("RGB")
        W_full, H_full = full.size
        x = torch.from_numpy(np.asarray(full.resize((LOC_SIZE, LOC_SIZE)), dtype=np.float32)
                             .transpose(2, 0, 1)[None] / 255.0).cuda()
        with torch.amp.autocast("cuda"):
            pred = torch.sigmoid(localiser(x))[0, 0].float().cpu().numpy() > 0.5
        if not pred.any():
            failures.append({"image_id": image_id, "reason": "localiser found no disc"})
            continue
        top, left, bottom, right = crops_mod.mask_box(pred)
        sy, sx = H_full / LOC_SIZE, W_full / LOC_SIZE
        disc_full = np.zeros((H_full, W_full), dtype=bool)
        disc_full[int(top * sy):int(bottom * sy), int(left * sx):int(right * sx)] = True
        box = crops_mod.square_box(disc_full, (H_full, W_full), margin=MARGIN)
        rows.append({"image_id": image_id, "height": H_full, "width": W_full,
                     "top": box[0], "left": box[1], "bottom": box[2], "right": box[3]})
        if n % 200 == 0:
            print(f"  {n}/{len(all_ids)}")

boxes = pd.DataFrame(rows)
BOX_FILE = os.path.join(WORK, "crops", "boxes.csv")
boxes.to_csv(BOX_FILE, index=False)
digest = hashlib.sha256(open(BOX_FILE, "rb").read()).hexdigest()
json.dump({"sha256": digest, "margin": MARGIN, "n": len(boxes), "gpu": GPU,
           "failures": failures},
          open(os.path.join(WORK, "crops", "boxes.sha256.json"), "w"), indent=2)
print(f"\n{len(boxes)} boxes frozen.  sha256 {digest[:16]}...")
print("failures:", failures if failures else "none")


In [ ]:
# --- PROVE the box holds every expert's disc ---------------------------------
def column(frame, name):
    return np.array([[int(v) for v in s.split(",")] for s in frame[name]], dtype=int)

check = boxes.merge(png, on="image_id", how="inner")
expert = np.stack([column(check, f"box_user{k}") for k in range(1, 8)])
union = np.stack([expert[:, :, 0].min(0), expert[:, :, 1].min(0),
                  expert[:, :, 2].max(0), expert[:, :, 3].max(0)], axis=1)
report = pd.DataFrame({
    "image_id": check.image_id,
    "slack_top": union[:, 0] - check.top.to_numpy(),
    "slack_left": union[:, 1] - check.left.to_numpy(),
    "slack_bottom": check.bottom.to_numpy() - union[:, 2],
    "slack_right": check.right.to_numpy() - union[:, 3],
})
report["contains_union"] = (report[["slack_top", "slack_left",
                                    "slack_bottom", "slack_right"]] >= 0).all(axis=1)
report["padding_fraction"] = [
    crops_mod.padding_fraction((r.top, r.left, r.bottom, r.right), (r.height, r.width))
    for r in check.itertuples()]
report.to_csv(os.path.join(WORK, "results", "containment.csv"), index=False)

padded = report[report.padding_fraction > 0]
print(f"eyes whose box extends past the image border: {len(padded)} / {len(report)}"
      f"  (max padding {100 * report.padding_fraction.max():.1f}% of the crop)")
print("   that padding is black, which is what a fundus surround already is\n")

# Amendment 4: padding is far commoner on Zeiss, which is the split we train on. It cancels
# in Delta_model (same box for every label source) but the asymmetry is reported, and
# notebook 02 repeats the gate on the unpadded evaluation eyes as a sensitivity check.
split_pad = (report.merge(inventory[["image_id", "split", "camera"]], on="image_id")
             .assign(padded=lambda d: d.padding_fraction > 0)
             .groupby(["split", "camera"]).padded.agg(["sum", "count"]))
split_pad["percent"] = (100 * split_pad["sum"] / split_pad["count"]).round(1)
print("padding by split - expect roughly 33% on Zeiss against 2-4% on Canon:")
print(split_pad.to_string())
print()
# How far is the frozen localiser from the official-merge disc it was trained on? The box has
# a median 96 px of slack at margin 1.8, so a few pixels are harmless - but the localiser
# trains on Zeiss and is applied to Canon, so any generalisation failure shows up HERE, as
# centre error concentrated on one camera, before it can quietly cost evaluation eyes.
official = np.array([[int(v) for v in s.split(",")] for s in check.box_official])
err = pd.DataFrame({
    "split": check.split.to_numpy(),          # png_measurements already carries the split
    "centre_dy": (official[:, 0] + official[:, 2]) / 2
                 - (check.top.to_numpy() + check.bottom.to_numpy()) / 2,
    "centre_dx": (official[:, 1] + official[:, 3]) / 2
                 - (check.left.to_numpy() + check.right.to_numpy()) / 2,
})
err["centre_error_px"] = np.hypot(err.centre_dy, err.centre_dx)
print("localiser box centre error vs the official-merge disc, by split (px):")
print(err.groupby("split").centre_error_px.describe()[["mean", "50%", "max"]].round(1).to_string())
print("   a camera-specific jump here means the localiser did not generalise Zeiss -> Canon\n")

bad = report[~report.contains_union]
print(f"eyes where the frozen box clips an expert's disc: {len(bad)} / {len(report)}")
if len(bad):
    print("   by split - failures concentrated on Canon would be a localiser problem, "
          "not a margin problem:")
    print("   " + bad.merge(inventory[["image_id", "split"]], on="image_id")
          .groupby("split").size().to_string().replace("\n", "\n   "))
print("\nsmallest slack in pixels (how close the box came to clipping):")
print(report[["slack_top", "slack_left", "slack_bottom", "slack_right"]].min().to_string())
if len(bad):
    display(bad.head(20))
    print("\nThese eyes are EXCLUDED and the count goes in the paper. They are not dropped "
          "quietly, and the margin is NOT raised now - that would be tuning after the fact.")


In [ ]:
# --- export the crops: one .npz per eye, 19 arrays, identical pixels ---------
CROP = train_mod.CONFIG.crop_size
keep = set(report.loc[report.contains_union, "image_id"])
train_set, val_set = set(split_ids["train"]), set(split_ids["val"])
manifest, t0 = [], time.time()

for n, (_, row) in enumerate(boxes.iterrows(), 1):
    image_id = row.image_id
    rel = os.path.join("crops", f"{image_id}.npz")
    if image_id not in keep:
        continue
    if os.path.exists(os.path.join(WORK, rel)):           # resume after a disconnect
        manifest.append({"image_id": image_id, "npz": rel,
                         "camera": inventory.loc[inventory.image_id == image_id,
                                                 "camera"].iloc[0],
                         "part": "train" if image_id in train_set
                                 else "val" if image_id in val_set else "eval"})
        continue
    folder = folder_of[image_id]
    jpg = next(f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg")))
    box = (int(row.top), int(row.left), int(row.bottom), int(row.right))

    def cut(arr, nearest=False):
        # crops_mod.crop pads where the box leaves the image. Plain slicing must NOT be used:
        # a negative index wraps around and pastes the opposite side of the retina into the crop.
        piece = crops_mod.crop(arr.astype(np.uint8) * 255 if nearest else arr, box)
        return np.array(Image.fromarray(piece).resize(
            (CROP, CROP), Image.NEAREST if nearest else Image.BILINEAR))

    arrays = {"image": cut(np.asarray(Image.open(os.path.join(folder, jpg)).convert("RGB")))
                       .astype(np.uint8)}
    cups, discs = [], []
    for k in range(1, 8):
        cup, disc = refuge.expert_masks(folder, image_id, k)
        cup, disc = cut(cup, nearest=True) > 127, cut(disc, nearest=True) > 127
        arrays[f"cup_user{k}"], arrays[f"disc_user{k}"] = cup, disc
        cups.append(cup); discs.append(disc)
    ocup, odisc = refuge.official_masks(folder, image_id)
    arrays["cup_official"] = cut(ocup, nearest=True) > 127
    arrays["disc_official"] = cut(odisc, nearest=True) > 127
    arrays["cup_majority"] = labels_mod.fuse(cups, "majority")
    arrays["disc_majority"] = labels_mod.fuse(discs, "majority")

    np.savez_compressed(os.path.join(WORK, rel), **arrays)
    manifest.append({"image_id": image_id, "npz": rel,
                     "camera": inventory.loc[inventory.image_id == image_id, "camera"].iloc[0],
                     "part": "train" if image_id in train_set
                             else "val" if image_id in val_set else "eval"})
    if n % 100 == 0:
        print(f"  {n}/{len(boxes)}   {time.time() - t0:5.0f}s")

manifest = pd.DataFrame(manifest)
manifest.to_csv(os.path.join(WORK, "crops", "manifest.csv"), index=False)
print(f"\n{len(manifest)} crops ready in {time.time() - t0:.0f}s")
print(manifest.groupby(["part", "camera"]).size().to_string())


In [ ]:
# --- the check that must pass before notebook 02 -----------------------------
# Same eye, two conventions. The IMAGE must be identical and the CUPS must not be.
sample = np.load(os.path.join(WORK, manifest.npz.iloc[0]))
liberal, conservative = sample["cup_user1"].sum(), sample["cup_user7"].sum()
print(f"image array      : {sample['image'].shape} {sample['image'].dtype}")
print(f"cup_user1 pixels : {liberal}")
print(f"cup_user7 pixels : {conservative}")
print(f"ratio            : {liberal / max(conservative, 1):.2f}x")
assert liberal != conservative, (
    "user1 and user7 produced the same cup - the export is wrong and nothing after this "
    "point means anything.")
print("\nOK: one image, different outlines. That difference is the whole experiment.")


In [ ]:
# --- Hugging Face: the archival copy ------------------------------------------
# The token comes from Colab Secrets (key icon in the left sidebar) under the name HF_TOKEN.
# It is never written into a cell, a file, or this repository.
!pip -q install -U huggingface_hub

from huggingface_hub import HfApi

HF_REPO = "Aduitya9/opiotica-whose-eyes"          # PRIVATE: it holds REFUGE-derived pixels
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as error:
    HF_TOKEN = None
    print("No HF_TOKEN in Colab Secrets:", error)

PUSH_TO_HF = HF_TOKEN is not None
hf = HfApi(token=HF_TOKEN) if PUSH_TO_HF else None
print("upload enabled" if PUSH_TO_HF else
      "upload DISABLED - add HF_TOKEN under Secrets to archive this run; "
      "everything still works locally and in Drive")


In [ ]:
# --- archive the frozen crops and the decisions that produced them -----------
# Why upload at all: the crops are the one expensive artefact (~45 min to rebuild) and they
# are what makes every later run comparable. Why PRIVATE: they contain REFUGE pixels.
if PUSH_TO_HF:
    for name in ("boxes.csv", "boxes.sha256.json", "manifest.csv"):
        hf.upload_file(path_or_fileobj=os.path.join(WORK, "crops", name),
                       path_in_repo=f"crops/{name}", repo_id=HF_REPO, repo_type="dataset",
                       commit_message=f"Frozen crop metadata: {name}")
    for name in ("inventory.csv", "png_measurements.csv", "crop_margin.json",
                 "containment.csv", "zeiss_train80_val20.json"):
        path = os.path.join(WORK, "results", name)
        if os.path.exists(path):
            hf.upload_file(path_or_fileobj=path, path_in_repo=f"results/{name}",
                           repo_id=HF_REPO, repo_type="dataset",
                           commit_message=f"Notebook 00/01 output: {name}")
    print("metadata uploaded\n")

    # the crops themselves: ~1,200 files, a few hundred MB. Slow; skip if you are short of time
    # and simply re-run this notebook later - nothing downstream needs them to be on the Hub.
    UPLOAD_CROPS = True
    if UPLOAD_CROPS:
        upload = getattr(hf, "upload_large_folder", None)
        if upload:
            upload(folder_path=os.path.join(WORK, "crops"), repo_id=HF_REPO,
                   repo_type="dataset", allow_patterns=["*.npz"])
        else:
            hf.upload_folder(folder_path=os.path.join(WORK, "crops"), path_in_repo="crops",
                             repo_id=HF_REPO, repo_type="dataset", allow_patterns=["*.npz"],
                             commit_message="Frozen crops: image + 18 masks per eye")
        print("crops uploaded")
    print(f"\nhttps://huggingface.co/datasets/{HF_REPO}")


## Done

`whose-eyes-work/crops/` now holds one `.npz` per eye: the image once, and eighteen masks —
seven experts × (cup, disc), plus majority and the official merge.

Next: `02_noise_floor_gate.ipynb`. Stay on the **same GPU type** you used here if you can;
notebook 02 refuses to report a gate computed across mixed hardware.
